# A Decade of Pitch Language
## Detecting Shifts in Marketing Adjective Usage in Kickstarter Campaigns
### Course Final Project — Data Analysis, 2026

**Research Questions:**
1. Is there a detectable changepoint in the use of specific adjective categories in Kickstarter pitches across 2014–2025?
2. Do such shifts correlate with campaign fundraising success?

**Data Source:** Web Robots (webrobots.io) — 6 Kickstarter snapshots (2016–2026)

**Pipeline:**
- Step 1: Build unified dataset (710k campaigns)
- Step 2: Extract adjectives with spaCy (4 categories)
- Step 3: Changepoint detection (PELT algorithm)
- Step 4: Logistic regression analysis
- Step 5: Visualizations

## Setup: Install Dependencies

In [ ]:
!pip install -q spacy ruptures scikit-learn scipy matplotlib seaborn
!python -m spacy download en_core_web_sm -q
print('All dependencies installed!')

## Step 1: Build Unified Dataset
Downloads 6 Kickstarter snapshots from Web Robots and builds a unified CSV with 710k campaigns.

In [ ]:
import urllib.request, gzip, json, os
import pandas as pd

SNAPSHOTS = [
    ('2016', 'https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2016-01-28T09_15_08_781Z.json.gz'),
    ('2018', 'https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2018-01-12T10_20_09_196Z.json.gz'),
    ('2020', 'https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2020-01-16T03_20_15_556Z.json.gz'),
    ('2022', 'https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2022-01-20T03_20_11_451Z.json.gz'),
    ('2024', 'https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2024-01-15T14_13_05_649Z.json.gz'),
    ('2026', 'https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2026-01-12T09_37_51_016Z.json.gz'),
]

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

def parse_record(line, snapshot_year):
    try:
        obj = json.loads(line)
        p = obj.get('data', obj)
        if not isinstance(p, dict): return None
        state = p.get('state', '')
        if state not in ('successful', 'failed'): return None
        blurb = (p.get('blurb') or '').strip()
        if len(blurb) < 10: return None
        cat = p.get('category') or {}
        return {
            'source': 'kickstarter', 'snapshot_year': snapshot_year,
            'id': p.get('id'), 'name': p.get('name', ''),
            'blurb': blurb, 'category': cat.get('name', ''),
            'parent_category': cat.get('parent_name', ''),
            'country': p.get('country', ''),
            'goal_usd': float(p.get('goal', 0) or 0),
            'pledged_usd': float(p.get('usd_pledged') or p.get('pledged') or 0),
            'backers': p.get('backers_count', 0),
            'state': state, 'success': 1 if state == 'successful' else 0,
            'launched_at': p.get('launched_at'), 'deadline': p.get('deadline'),
        }
    except: return None

all_frames = []
for year, url in SNAPSHOTS:
    gz = f'data/raw/kickstarter_{year}.json.gz'
    if not os.path.exists(gz):
        print(f'[{year}] Downloading...')
        urllib.request.urlretrieve(url, gz)
    records = []
    with gzip.open(gz, 'rt', encoding='utf-8', errors='replace') as f:
        for line in f:
            r = parse_record(line.strip(), year)
            if r: records.append(r)
    df_y = pd.DataFrame(records).drop_duplicates(subset='id')
    print(f'[{year}] {len(df_y):,} projects | success={df_y["success"].mean():.1%}')
    all_frames.append(df_y)

df = pd.concat(all_frames, ignore_index=True)
df['launched_date'] = pd.to_datetime(df['launched_at'], unit='s', errors='coerce')
df['deadline_date'] = pd.to_datetime(df['deadline'], unit='s', errors='coerce')
df['duration_days'] = (df['deadline_date'] - df['launched_date']).dt.days
df['blurb_len'] = df['blurb'].str.len()
df.to_csv('data/processed/kickstarter_unified.csv', index=False)
print(f'\nTotal: {len(df):,} projects saved.')

## Step 2: Extract Adjectives with spaCy
Classifies adjectives into 4 semantic categories: Superlative, Urgency/Scarcity, Emotional, Quality/Technical.

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm', disable=['ner', 'parser'])

SUPERLATIVES     = {'best','greatest','largest','fastest','highest','biggest','first','ultimate','top','finest','premier','leading','unmatched','unrivaled','unparalleled','supreme','perfect','ideal','only','sole'}
URGENCY_SCARCITY = {'limited','exclusive','rare','scarce','urgent','immediate','critical','essential','vital','crucial','last','final','early','special','unique','temporary','short','few','remaining','expiring','closing'}
EMOTIONAL        = {'amazing','incredible','awesome','fantastic','wonderful','beautiful','stunning','breathtaking','inspiring','passionate','exciting','thrilling','magical','extraordinary','remarkable','unforgettable','powerful','heartfelt','moving','joyful','fun','hilarious','adorable','lovely','charming','delightful','epic','bold','brave','daring','creative','innovative','revolutionary','groundbreaking'}
QUALITY_TECHNICAL= {'professional','premium','quality','durable','reliable','robust','advanced','smart','efficient','effective','precise','accurate','secure','safe','certified','tested','proven','engineered','designed','crafted','handmade','custom','sustainable','organic','natural','authentic','original','genuine','lightweight','compact','portable','wireless','digital','modular','waterproof','eco-friendly'}

def categorize(word):
    w = word.lower()
    cats = []
    if w in SUPERLATIVES:      cats.append('superlative')
    if w in URGENCY_SCARCITY:  cats.append('urgency_scarcity')
    if w in EMOTIONAL:         cats.append('emotional')
    if w in QUALITY_TECHNICAL: cats.append('quality_technical')
    return cats or ['other']

df = pd.read_csv('data/processed/kickstarter_unified.csv', low_memory=False)
df['launched_date'] = pd.to_datetime(df['launched_date'], errors='coerce')
df['launch_year']   = df['launched_date'].dt.year
df = df[df['launch_year'].between(2014, 2025)].copy()
df = df[df['blurb'].notna() & (df['blurb'].str.len() >= 10)].copy()
df = df.drop_duplicates(subset='id').reset_index(drop=True)
print(f'Processing {len(df):,} projects...')

BATCH = 2000
results = []
for i in range(0, len(df), BATCH):
    batch = df.iloc[i:i+BATCH]
    docs  = list(nlp.pipe(batch['blurb'].tolist(), batch_size=256))
    for j, doc in enumerate(docs):
        row  = batch.iloc[j]
        adjs = [t.lemma_.lower() for t in doc if t.pos_ == 'ADJ' and t.is_alpha and len(t.text) > 2]
        cnt  = {'superlative':0,'urgency_scarcity':0,'emotional':0,'quality_technical':0}
        for a in adjs:
            for c in categorize(a):
                if c in cnt: cnt[c] += 1
        wc = max(len([t for t in doc if t.is_alpha]), 1)
        results.append({
            'id': row['id'], 'launch_year': row['launch_year'],
            'success': row['success'], 'category': row['category'],
            'parent_category': row['parent_category'],
            'goal_usd': row['goal_usd'], 'blurb_len': row['blurb_len'],
            'word_count': wc, 'total_adj': len(adjs),
            'adj_rate': len(adjs)/wc,
            'rate_superlative':       cnt['superlative']/wc,
            'rate_urgency_scarcity':  cnt['urgency_scarcity']/wc,
            'rate_emotional':         cnt['emotional']/wc,
            'rate_quality_technical': cnt['quality_technical']/wc,
        })
    if (i // BATCH) % 20 == 0:
        print(f'  {min(i+BATCH, len(df)):,} / {len(df):,}')

adj_df = pd.DataFrame(results)
adj_df.to_csv('data/processed/kickstarter_adjectives.csv', index=False)
print('Done! Saved to data/processed/kickstarter_adjectives.csv')

## Step 3: Temporal Analysis & Changepoint Detection

In [ ]:
import ruptures as rpt
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

adj_df = pd.read_csv('data/processed/kickstarter_adjectives.csv')
adj_df = adj_df[adj_df['launch_year'].between(2014, 2025)]

ADJ_COLS = ['rate_superlative','rate_urgency_scarcity','rate_emotional','rate_quality_technical']
LABELS   = ['Superlative','Urgency/Scarcity','Emotional','Quality/Technical']
COLORS   = ['#2196F3','#F44336','#4CAF50','#FF9800']

yearly = adj_df.groupby('launch_year')[ADJ_COLS + ['success']].agg(
    rate_superlative=('rate_superlative','mean'),
    rate_urgency_scarcity=('rate_urgency_scarcity','mean'),
    rate_emotional=('rate_emotional','mean'),
    rate_quality_technical=('rate_quality_technical','mean'),
    success_rate=('success','mean'),
    n_projects=('success','count'),
).reset_index().sort_values('launch_year')

print('Yearly adjective rates:')
print(yearly[['launch_year'] + ADJ_COLS + ['success_rate','n_projects']].to_string(index=False))

In [ ]:
# ── Figure 1: Adjective Trends ────────────────────────────────────────────
plt.rcParams.update({'font.family':'serif','font.size':11,'axes.spines.top':False,'axes.spines.right':False})

fig, ax = plt.subplots(figsize=(10, 5))
for col, label, color in zip(ADJ_COLS, LABELS, COLORS):
    ax.plot(yearly['launch_year'], yearly[col]*100, marker='o', linewidth=2.5,
            markersize=6, label=label, color=color)
ax.set_xlabel('Launch Year'); ax.set_ylabel('Adjective Usage Rate (% of words)')
ax.set_title('Marketing Adjective Usage in Kickstarter Pitches (2014–2025)', fontsize=13)
ax.legend(loc='upper right'); ax.grid(axis='y', alpha=0.3)
ax.set_xticks(yearly['launch_year'])
plt.tight_layout(); plt.show()
print('Figure 1: Adjective trends over time')

In [ ]:
# ── Changepoint Detection ─────────────────────────────────────────────────
CAT_KEYS = ['superlative','urgency_scarcity','emotional','quality_technical']
cp_results = {}

print('--- Changepoint Detection Results ---')
for cat, col in zip(CAT_KEYS, ADJ_COLS):
    signal = yearly[col].values.reshape(-1,1)
    sig_norm = (signal - signal.mean()) / (signal.std() + 1e-10)
    model = rpt.Pelt(model='rbf', min_size=2, jump=1).fit(sig_norm)
    bps = model.predict(pen=2.0)
    cp_years = [int(yearly['launch_year'].iloc[bp-1]) for bp in bps[:-1]]
    cp_results[cat] = {'changepoint_years': cp_years, 'col': col}
    if cp_years:
        before = adj_df[adj_df['launch_year'] <  cp_years[0]][col]
        after  = adj_df[adj_df['launch_year'] >= cp_years[0]][col]
        t_stat, p_val = stats.ttest_ind(before, after)
        cp_results[cat].update({'before_mean':float(before.mean()),'after_mean':float(after.mean()),'p_value':float(p_val)})
        sig = '***' if p_val<0.001 else ('**' if p_val<0.01 else ('*' if p_val<0.05 else 'ns'))
        direction = '↑' if after.mean() > before.mean() else '↓'
        print(f'  {cat:20s}: CP={cp_years} before={before.mean():.5f} after={after.mean():.5f} {direction} p={p_val:.4f} {sig}')
    else:
        print(f'  {cat:20s}: no significant changepoint')

In [ ]:
# ── Figure 2: Changepoint Detection Plot ──────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, (cat, col, label, color) in enumerate(zip(CAT_KEYS, ADJ_COLS, LABELS, COLORS)):
    ax   = axes[i]
    yrs  = yearly['launch_year'].values
    vals = yearly[col].values * 100
    ax.plot(yrs, vals, color=color, linewidth=2.5, marker='o', markersize=6)
    ax.fill_between(yrs, vals, alpha=0.08, color=color)
    cp_years = cp_results.get(cat, {}).get('changepoint_years', [])
    sig = ''
    if cp_years:
        p = cp_results[cat].get('p_value', 1)
        sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else ''))
        ax.axvline(x=cp_years[0], color='red', linestyle='--', linewidth=2, alpha=0.85)
        ax.text(cp_years[0]+0.15, max(vals)*0.97, f'CP: {cp_years[0]}', color='red', fontsize=9, va='top')
    ax.set_title(f'{label}  {sig}', fontsize=11)
    ax.set_xlabel('Year'); ax.set_ylabel('Rate (%)')
    ax.set_xticks(yrs[::2]); ax.grid(axis='y', alpha=0.3)

plt.suptitle('Changepoint Detection in Adjective Category Usage\n(red dashed = detected changepoint, *** p<0.001)', fontsize=12)
plt.tight_layout(); plt.show()
print('Figure 2: Changepoint detection')

## Step 4: Success Analysis

In [ ]:
# ── Before/After Changepoint: Success Rate ────────────────────────────────
print('--- Before/After Changepoint: Success Rate ---')
for cat, col in zip(CAT_KEYS, ADJ_COLS):
    cp_years = cp_results.get(cat, {}).get('changepoint_years', [])
    if not cp_years: continue
    cp = cp_years[0]
    before_df = adj_df[adj_df['launch_year'] < cp]
    after_df  = adj_df[adj_df['launch_year'] >= cp]
    hi_b = before_df[before_df[col] > before_df[col].median()]['success'].mean()
    lo_b = before_df[before_df[col] <= before_df[col].median()]['success'].mean()
    hi_a = after_df[after_df[col] > after_df[col].median()]['success'].mean()
    lo_a = after_df[after_df[col] <= after_df[col].median()]['success'].mean()
    print(f'\n{cat} (CP={cp}):')
    print(f'  Before: high={hi_b:.1%}  low={lo_b:.1%}  diff={hi_b-lo_b:+.1%}')
    print(f'  After:  high={hi_a:.1%}  low={lo_a:.1%}  diff={hi_a-lo_a:+.1%}')

In [ ]:
# ── Figure 3: Success by Adj Usage ───────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for i, (cat, col, label, color) in enumerate(zip(CAT_KEYS, ADJ_COLS, LABELS, COLORS)):
    ax  = axes[i]
    med = adj_df[col].median()
    adj_df['_grp'] = adj_df[col].apply(lambda x: 'High' if x > med else 'Low')
    sr  = adj_df.groupby('_grp')['success'].mean() * 100
    sr  = sr.reindex(['Low','High'])
    bars = ax.bar(['Low','High'], sr.values, color=['#CCCCCC', color], width=0.5)
    for bar, val in zip(bars, sr.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.8,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(label, fontsize=10)
    ax.set_ylabel('Success Rate (%)' if i==0 else '')
    ax.set_ylim(0, 90); ax.grid(axis='y', alpha=0.3)

adj_df.drop(columns='_grp', inplace=True)
plt.suptitle('Campaign Success Rate by Adjective Category Usage (High vs. Low)', fontsize=12)
plt.tight_layout(); plt.show()
print('Figure 3: Success rate by adjective usage')

## Step 5: Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.dummy import DummyClassifier

df_reg = adj_df.dropna(subset=['success','goal_usd','blurb_len']).copy()
df_reg = df_reg[df_reg['goal_usd'] > 0].copy()
df_reg['log_goal'] = np.log1p(df_reg['goal_usd'])

top_cats = df_reg['parent_category'].value_counts().nlargest(10).index
df_reg['parent_cat'] = df_reg['parent_category'].where(df_reg['parent_category'].isin(top_cats), 'Other')
cat_dummies = pd.get_dummies(df_reg['parent_cat'], prefix='cat', drop_first=True)

y = df_reg['success']
CONTROLS = ['log_goal','blurb_len','adj_rate']
X_base = pd.concat([df_reg[CONTROLS], cat_dummies], axis=1).fillna(0)
X_full = pd.concat([df_reg[ADJ_COLS + CONTROLS], cat_dummies], axis=1).fillna(0)

sc_b = StandardScaler(); X_base_s = sc_b.fit_transform(X_base)
sc_f = StandardScaler(); X_full_s = sc_f.fit_transform(X_full)

auc_dummy = cross_val_score(DummyClassifier(), X_base_s, y, cv=5, scoring='roc_auc').mean()
auc_base  = cross_val_score(LogisticRegression(max_iter=1000), X_base_s, y, cv=5, scoring='roc_auc').mean()
auc_full  = cross_val_score(LogisticRegression(max_iter=1000), X_full_s, y, cv=5, scoring='roc_auc').mean()

print(f'Dummy baseline AUC : {auc_dummy:.4f}')
print(f'Baseline AUC       : {auc_base:.4f}')
print(f'Full model AUC     : {auc_full:.4f}')
print(f'Improvement        : +{auc_full-auc_base:.4f}')

lr = LogisticRegression(max_iter=1000)
lr.fit(X_full_s, y)
feat_names = ADJ_COLS + CONTROLS + list(cat_dummies.columns)
coef_df = pd.DataFrame({'feature': feat_names, 'coefficient': lr.coef_[0]})
print('\nAdjective coefficients:')
print(coef_df[coef_df['feature'].isin(ADJ_COLS)].sort_values('coefficient', ascending=False).to_string(index=False))

In [ ]:
# ── Figure 4: Regression Coefficients ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

adj_coefs = coef_df[coef_df['feature'].isin(ADJ_COLS)].copy()
adj_coefs['label'] = adj_coefs['feature'].map(dict(zip(ADJ_COLS, LABELS)))
adj_coefs['cat']   = adj_coefs['feature'].str.replace('rate_','')
adj_coefs = adj_coefs.sort_values('coefficient')
colors_coef = [COLORS[CAT_KEYS.index(c)] for c in adj_coefs['cat']]

axes[0].barh(adj_coefs['label'], adj_coefs['coefficient'], color=colors_coef, height=0.5)
axes[0].axvline(x=0, color='black', linewidth=0.8)
axes[0].set_xlabel('Logistic Regression Coefficient (standardized)')
axes[0].set_title('Effect of Adjective Categories on Success')
axes[0].grid(axis='x', alpha=0.3)
for i, (_, row) in enumerate(adj_coefs.iterrows()):
    axes[0].text(row['coefficient']+0.001, i, f"{row['coefficient']:+.3f}", va='center', fontsize=9)

# AUC comparison
models = ['Dummy\nBaseline', 'No Adjectives', 'With\nAdjectives']
aucs   = [auc_dummy, auc_base, auc_full]
cols   = ['#DDDDDD','#90CAF9','#1565C0']
bars   = axes[1].bar(models, aucs, color=cols, width=0.45)
for bar, val in zip(bars, aucs):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_ylabel('AUC-ROC (5-fold CV)')
axes[1].set_title('Predictive Performance')
axes[1].set_ylim(0.45, min(1.0, max(aucs)+0.06))
axes[1].axhline(y=0.5, color='gray', linestyle=':', linewidth=1)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()
print('Figure 4: Regression coefficients and AUC comparison')

## Summary of Findings

| Finding | Result |
|---------|--------|
| Corpus size | 278,628 campaigns (2014–2025) |
| Changepoint: Superlative | 2019 — 36% decline (p<0.001) |
| Changepoint: Urgency/Scarcity | 2016 — 31% increase (p<0.001) |
| High superlative → success (before CP) | +8.8pp advantage |
| High superlative → success (after CP) | +5.8pp advantage |
| Strongest predictor | Superlative (β=+0.054) |
| AUC improvement from adj features | +0.0001 |

**Conclusion:** Crowdfunding pitch language has undergone measurable rhetorical evolution over the past decade. Creators shifted away from absolute superlatives (declining from 2019) toward urgency/scarcity framing (rising from 2016). Both strategies are positively associated with campaign success, though the advantage of superlative language has narrowed as the strategy has diffused.